# Preparación de datos SCINCE

Carga y consolida los datos del SCINCE 2020 y SCINCE 2010 en tablas planas
por nivel geográfico (municipal, estatal, nacional).

**Este notebook se ejecuta una sola vez** (o cuando se actualice la fuente de datos).
Su salida — cuatro archivos CSV en `datos/tablas/` — es el insumo de los notebooks
de análisis y visualización.

**Fuente:** Censo de Población y Vivienda 2020 y 2010, INEGI (SCINCE).

> **Requisito para TCMA:** copia (o crea un enlace simbólico del) directorio `00`
> de tu instalación de SCINCE 2010 en `datos/scince_2010/00` antes de ejecutar.
> Si la ruta no existe, la columna `TCMA_2010_2020` quedará como `NaN`.

In [5]:
import geopandas as gpd
import pandas as pd
import pyogrio
from pathlib import Path

## 1. Configuración

In [6]:
# ── SCINCE 2020 ───────────────────────────────────────────────────────────────
SCINCE_2020_BASE = Path("C:/SCINCE 2020/00_NAL")
CART_2020        = SCINCE_2020_BASE / "cartografia"
TABLAS_2020      = SCINCE_2020_BASE / "tablas"
RUTA_DESC        = SCINCE_2020_BASE / "Descriptores.xlsx"

# ── SCINCE 2010 ───────────────────────────────────────────────────────────────
SCINCE_2010_BASE = Path("../datos/scince_2010/00")

# ── Salida ────────────────────────────────────────────────────────────────────
RUTA_SALIDA = Path("../datos/tablas")

## 2. Carga de datos SCINCE 2020

In [7]:
def leer_dbf(ruta, nivel="municipal"):
    pad = {"municipal": 5, "estatal": 2, "nacional": 2}[nivel]
    df = pyogrio.read_dataframe(str(ruta))
    df["CVEGEO"] = df["CVEGEO"].astype(str).str.zfill(pad)
    df.columns = [c.upper() for c in df.columns]
    return df.drop(columns=["GEOMETRY"], errors="ignore")

# ── Municipal ─────────────────────────────────────────────────────────────────
mun_shp = gpd.read_file(str(CART_2020 / "municipal.shp"))
mun_shp["CVEGEO"] = mun_shp["CVEGEO"].astype(str).str.zfill(5)
mun_shp.columns = [c.upper() for c in mun_shp.columns]

df_mun = (
    mun_shp.drop(columns=["GEOMETRY"])
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_municipal_migracion.dbf"),
           on="CVEGEO", how="left", suffixes=("", "_m"))
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_municipal_discapacidad.dbf"),
           on="CVEGEO", how="left", suffixes=("", "_d"))
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_municipal_etnicidad.dbf"),
           on="CVEGEO", how="left", suffixes=("", "_e"))
)

# ── Estatal ───────────────────────────────────────────────────────────────────
est_shp = gpd.read_file(str(CART_2020 / "estatal.shp"))
est_shp["CVEGEO"] = est_shp["CVEGEO"].astype(str).str.zfill(2)
est_shp.columns = [c.upper() for c in est_shp.columns]

df_est = (
    est_shp.drop(columns=["GEOMETRY"])
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_estatal_migracion.dbf",    "estatal"),
           on="CVEGEO", how="left", suffixes=("", "_m"))
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_estatal_discapacidad.dbf", "estatal"),
           on="CVEGEO", how="left", suffixes=("", "_d"))
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_estatal_etnicidad.dbf",    "estatal"),
           on="CVEGEO", how="left", suffixes=("", "_e"))
)

# ── Nacional ──────────────────────────────────────────────────────────────────
nal_shp = gpd.read_file(str(CART_2020 / "nacional.shp"))
nal_shp["CVEGEO"] = nal_shp["CVEGEO"].astype(str).str.zfill(2)
nal_shp.columns = [c.upper() for c in nal_shp.columns]

df_nal = (
    nal_shp.drop(columns=["GEOMETRY"])
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_nacional_migracion.dbf",    "nacional"),
           on="CVEGEO", how="left", suffixes=("", "_m"))
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_nacional_discapacidad.dbf", "nacional"),
           on="CVEGEO", how="left", suffixes=("", "_d"))
    .merge(leer_dbf(TABLAS_2020 / "cpv2020_nacional_etnicidad.dbf",    "nacional"),
           on="CVEGEO", how="left", suffixes=("", "_e"))
)

print(f"Municipal 2020 : {len(df_mun):,} municipios · {df_mun.shape[1]} columnas")
print(f"Estatal   2020 : {len(df_est):,} entidades  · {df_est.shape[1]} columnas")
print(f"Nacional  2020 : {len(df_nal):,} registro   · {df_nal.shape[1]} columnas")

Municipal 2020 : 2,469 municipios · 398 columnas
Estatal   2020 : 32 entidades  · 395 columnas
Nacional  2020 : 1 registro   · 395 columnas


## 3. Carga de datos SCINCE 2010 y cálculo de TCMA

Se agrega la columna `TCMA_2010_2020` a los tres niveles geográficos.

> **TCMA** = (P₂₀₂₀ / P₂₀₁₀)^(1/10) − 1, expresada en porcentaje.

In [8]:
def _agregar_tcma(df_2020, shp_2010, nivel="municipal"):
    pad = {"municipal": 5, "estatal": 2, "nacional": 2}[nivel]
    df10 = gpd.read_file(str(shp_2010))
    df10["CVEGEO"] = df10["CVEGEO"].astype(str).str.zfill(pad)
    df10.columns = [c.upper() for c in df10.columns]
    df10 = df10[["CVEGEO", "POB1"]].rename(columns={"POB1": "POB1_2010"})
    merged = df_2020.merge(df10, on="CVEGEO", how="left")
    merged["TCMA_2010_2020"] = (
        (merged["POB1"] / merged["POB1_2010"]) ** (1 / 10) - 1
    ) * 100
    return merged

if SCINCE_2010_BASE.exists():
    df_mun = _agregar_tcma(df_mun, SCINCE_2010_BASE / "municipal.shp", "municipal")
    df_est = _agregar_tcma(df_est, SCINCE_2010_BASE / "estatal.shp",   "estatal")
    df_nal = _agregar_tcma(df_nal, SCINCE_2010_BASE / "nacional.shp",  "nacional")
    print("TCMA 2010-2020 calculada para los tres niveles.")
    print(f"  Ejemplo nacional: {df_nal['TCMA_2010_2020'].values[0]:.2f}%")
else:
    for df in (df_mun, df_est, df_nal):
        df["TCMA_2010_2020"] = float("nan")
    print(f"ADVERTENCIA: {SCINCE_2010_BASE} no encontrado.")
    print("  Copia el directorio 00 de SCINCE 2010 a datos/scince_2010/00 y re-ejecuta.")

ADVERTENCIA: ..\datos\scince_2010\00 no encontrado.
  Copia el directorio 00 de SCINCE 2010 a datos/scince_2010/00 y re-ejecuta.


## 4. Exportación

In [9]:
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

df_mun.to_csv(RUTA_SALIDA / "municipal_2020.csv", index=False, encoding="utf-8-sig")
df_est.to_csv(RUTA_SALIDA / "estatal_2020.csv",   index=False, encoding="utf-8-sig")
df_nal.to_csv(RUTA_SALIDA / "nacional_2020.csv",  index=False, encoding="utf-8-sig")

# Descriptores: incluye etiqueta manual para TCMA (no está en Descriptores.xlsx)
df_desc = pd.read_excel(RUTA_DESC, usecols=["CAMPO", "DESCRIP"])
df_desc["CAMPO"] = df_desc["CAMPO"].str.upper()
tcma_row = pd.DataFrame([{
    "CAMPO":  "TCMA_2010_2020",
    "DESCRIP": "Tasa de Crecimiento Media Anual 2010-2020 (%)",
}])
df_desc = pd.concat([df_desc, tcma_row], ignore_index=True)
df_desc.to_csv(RUTA_SALIDA / "descriptores.csv", index=False, encoding="utf-8-sig")

print(f"Archivos exportados en {RUTA_SALIDA.resolve()}")
for fname in ("municipal_2020.csv", "estatal_2020.csv", "nacional_2020.csv", "descriptores.csv"):
    p = RUTA_SALIDA / fname
    print(f"  {fname:<28} {p.stat().st_size / 1024:>8.0f} KB")

Archivos exportados en C:\Users\COMIMSA\repositorios\analisisDemografico\datos\tablas
  municipal_2020.csv               5166 KB
  estatal_2020.csv                   82 KB
  nacional_2020.csv                   6 KB
  descriptores.csv                   58 KB
